# 01 - Ingestion and Parsing
## Shakespeare Evidence Desk


## Why This Stage

*In general — what ingestion and parsing are and why any RAG system needs them.* Source documents arrive as messy, unstructured files — PDFs, HTML, scanned pages, or raw text laced with headers, footers, and boilerplate. A machine sees no meaningful structure in them, and without structure there is nothing to cite, filter, or retrieve by precisely. Ingestion and parsing is the stage that turns those raw files into clean, consistently structured units with their source location preserved.

It comes first in the pipeline because every later stage inherits its output: chunking groups these units, embeddings encode them, retrieval filters on their metadata, and final answers cite them. Errors introduced here — a lost heading, boilerplate leaking into content, a broken encoding — do not stay local; they silently corrupt everything built on top. That is why parsing is treated as a deliberate, measured decision rather than a quick cleanup script.


- **Goal:** turn the ten verified raw play texts into clean, structured document blocks (act / scene / speaker / speech / stage direction) that later stages can chunk and embed.
- **Loads from Step 00:** the setup handoff (`.artifacts/setup-handoff.json`) and labeled questions (`.artifacts/eval-questions.json`). We read those artifacts, not the previous kernel's variables.
- **The experiment:** compare a **deterministic** parser (regular expressions over the known Gutenberg layout) with a **model-assisted** parser (the chat deployment), then select one with a justification.
- **Preserve provenance:** keep the original source text and Gutenberg administrative notices; isolate the notices from retrieval content instead of deleting the raw file.
- **Not later stages:** this notebook does not chunk, embed, retrieve, or answer. It exports parsed blocks for notebook 02 (chunking).


## 1. Load the Setup Handoff and Labeled Questions

- **Purpose:** start from Step 00's verified selection instead of hidden kernel state. The handoff names the exact corpus snapshot; the questions define what we will later be measured against.
- **What the code does:** locates the repository root (using `.env.example` as a marker), reads both JSON artifacts, and validates they contain the expected ten documents and a labeled question schema.
- **Fails loudly if missing:** if `.artifacts/setup-handoff.json` is absent, run notebook 00 first — this notebook cannot invent the corpus location.
- **Expected result:** confirmation of the corpus prefix, ten documents, and the development/held-out question counts. No Azure call happens yet.


In [ ]:
import json
import re
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
handoff_path = ARTIFACTS / 'setup-handoff.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not handoff_path.is_file():
    raise RuntimeError('Missing .artifacts/setup-handoff.json. Run notebook 00 first.')

handoff = json.loads(handoff_path.read_text(encoding='utf-8'))
documents = handoff['documents']
corpus_prefix = handoff['corpus_prefix']
storage = handoff['storage']
if len(documents) != 10:
    raise RuntimeError('Expected ten documents in the setup handoff.')

evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

print(f'Corpus prefix: {corpus_prefix}')
print(f'Documents: {len(documents)} plays from snapshot in container "{storage["container"]}".')
print(f'Questions: {len(evaluation_questions.get("development", []))} development, '
      f'{len(evaluation_questions.get("heldout", []))} held-out.')


## 2. Fetch the Corpus Text (Cached Locally)

- **Purpose:** download the exact verified blobs named in the handoff so parsing runs on the same bytes that passed Step 00's checksum checks.
- **Identity auth:** uses Azure CLI with `--auth-mode login`, matching Step 00. Run `az login` first if needed; no account keys are used.
- **Local cache:** texts are written under the ignored `.artifacts/corpus-cache/<prefix>/` and reused on reruns, so you do not re-download every time.
- **Read-only:** this step only reads blobs. It does not modify, re-upload, or parse anything yet.
- **Expected result:** ten local text files and a `raw_texts` dictionary keyed by document id, with their character counts.


In [ ]:
corpus_cache = ARTIFACTS / 'corpus-cache' / corpus_prefix
corpus_cache.mkdir(parents=True, exist_ok=True)


def download_blob(blob_name, destination):
    # Identity-based blob download, consistent with notebook 00.
    subprocess.run(
        [
            'az', 'storage', 'blob', 'download',
            '--account-name', storage['account'],
            '--container-name', storage['container'],
            '--name', blob_name, '--file', str(destination),
            '--auth-mode', 'login', '--only-show-errors',
        ],
        check=True, capture_output=True, text=True,
    )


raw_texts = {}
for document in documents:
    local_path = corpus_cache / (document['id'] + '.txt')
    if not local_path.is_file():
        download_blob(document['blob_name'], local_path)
    raw_texts[document['id']] = local_path.read_text(encoding='utf-8')

for document in documents:
    print(f"{document['title']}: {len(raw_texts[document['id']]):,} characters")
print(f'Cached {len(raw_texts)} plays under {corpus_cache.relative_to(ROOT)}.')


## 3. Isolate Gutenberg Administrative Notices

- **Purpose:** separate Project Gutenberg's license header and footer from the actual play, so those notices never enter retrieval while the raw file stays intact.
- **How:** Gutenberg wraps each work between `*** START OF THE PROJECT GUTENBERG EBOOK ... ***` and `*** END OF ... ***`. We split on those markers into `header_notice`, `body`, and `footer_notice`.
- **Front matter stays with the body for now:** the title, author, contents table, and dramatis personae sit just inside the start marker. The structural parser (next step) skips to the first real `ACT` heading, so the table of contents is not mistaken for scenes.
- **Preservation check:** we assert both markers exist for every play; a missing marker means an unexpected edition and is raised rather than silently guessed.
- **Expected result:** per-play body character counts (smaller than the raw totals) and confirmation that notices were isolated, not discarded.


In [ ]:
START_MARKER = re.compile(r'^\*\*\* START OF THE PROJECT GUTENBERG EBOOK .*\*\*\*\s*$', re.MULTILINE)
END_MARKER = re.compile(r'^\*\*\* END OF THE PROJECT GUTENBERG EBOOK .*\*\*\*\s*$', re.MULTILINE)


def split_gutenberg(text):
    start = START_MARKER.search(text)
    end = END_MARKER.search(text)
    if not start or not end:
        raise RuntimeError('Expected Project Gutenberg start/end markers were not found.')
    return {
        'header_notice': text[:start.end()],
        'body': text[start.end():end.start()].strip('\n'),
        'footer_notice': text[end.start():],
    }


bodies = {}
for document in documents:
    parts = split_gutenberg(raw_texts[document['id']])
    bodies[document['id']] = parts['body']
    raw_len = len(raw_texts[document['id']])
    body_len = len(parts['body'])
    print(f"{document['title']}: body {body_len:,} chars "
          f"({raw_len - body_len:,} chars of notices isolated)")
print('Gutenberg notices isolated from all ten bodies; raw files unchanged.')


## 4. Deterministic Parser (Regular Expressions)

- **Idea:** the Gutenberg editions follow a consistent layout, so plain rules can extract structure cheaply and repeatably with no model cost.
- **Rules (grounded in the actual text):**
  - **Act heading:** a line like `ACT I` in capitals at the left margin.
  - **Scene heading:** a line like `SCENE I. A platform before the Castle.`
  - **Speaker:** an all-capital name line ending in a period, for example `HAMLET.` or `FIRST PLAYER.`
  - **Stage direction:** bracketed italics such as `[_Exit Ghost._]` or lines starting with `Enter` / `Exit` / `Exeunt`.
  - **Speech:** the lines following a speaker until the next speaker, stage direction, or heading.
- **Front matter skipped:** parsing starts at the first real `ACT` heading, so the contents table and dramatis personae are not misread as scenes. They are retained separately as `front_matter`.
- **Provenance:** every block records its source line span within the body, so later stages and labels can point back to the original text.
- **Honest expectation:** rules will miss edge cases (songs, letters read aloud, unusual speaker tags). The measurement step quantifies those misses rather than assuming the parser is perfect.
- **Expected result:** per-play counts of acts, scenes, speeches, and unique speakers, plus a line-coverage percentage.


In [ ]:
ACT_RE = re.compile(r'^ACT ([IVXLC]+)\.?\s*$')
SCENE_RE = re.compile(r'^\s*SCENE ([IVXLC]+)\.\s*(.*)$')
SPEAKER_RE = re.compile(r"^([A-Z][A-Z’'.-]*(?: [A-Z][A-Z’'.-]*)*)\.$")
STAGE_RE = re.compile(r'^(\[_.*)|^(Enter|Exit|Exeunt|Re-enter|Re-enters|Flourish|Alarum)\b')


def parse_play_deterministic(work_id, title, body):
    lines = body.split('\n')
    # Front matter ends at the first real ACT heading at the left margin.
    body_start = next((i for i, line in enumerate(lines) if ACT_RE.match(line)), 0)
    front_matter = '\n'.join(lines[:body_start]).strip()

    blocks = []
    act = scene = None
    current = None  # an open speech block being accumulated

    def close_speech():
        nonlocal current
        if current is not None:
            current['text'] = '\n'.join(current['_lines']).strip()
            current.pop('_lines')
            if current['text']:
                blocks.append(current)
        current = None

    for index in range(body_start, len(lines)):
        line = lines[index]
        source_line = index + 1  # 1-based within the body
        act_match = ACT_RE.match(line)
        scene_match = SCENE_RE.match(line)
        if act_match:
            close_speech()
            act = act_match.group(1)
            scene = None
            continue
        if scene_match:
            close_speech()
            scene = scene_match.group(1)
            blocks.append({'work_id': work_id, 'work': title, 'act': act, 'scene': scene,
                           'type': 'scene_heading', 'speaker': None,
                           'text': scene_match.group(2).strip(), 'source_line': source_line})
            continue
        if STAGE_RE.match(line.strip()):
            close_speech()
            blocks.append({'work_id': work_id, 'work': title, 'act': act, 'scene': scene,
                           'type': 'stage_direction', 'speaker': None,
                           'text': line.strip().strip('[]_'), 'source_line': source_line})
            continue
        speaker_match = SPEAKER_RE.match(line.strip())
        if speaker_match:
            close_speech()
            current = {'work_id': work_id, 'work': title, 'act': act, 'scene': scene,
                       'type': 'speech', 'speaker': speaker_match.group(1),
                       'source_line': source_line, '_lines': []}
            continue
        if current is not None:
            current['_lines'].append(line)
    close_speech()
    return {'front_matter': front_matter, 'blocks': blocks}


def summarize(parsed):
    blocks = parsed['blocks']
    speeches = [b for b in blocks if b['type'] == 'speech']
    return {
        'acts': len({b['act'] for b in blocks if b['act']}),
        'scenes': len([b for b in blocks if b['type'] == 'scene_heading']),
        'speeches': len(speeches),
        'stage_directions': len([b for b in blocks if b['type'] == 'stage_direction']),
        'speakers': len({b['speaker'] for b in speeches}),
    }


deterministic_parsed = {}
for document in documents:
    parsed = parse_play_deterministic(document['id'], document['title'], bodies[document['id']])
    deterministic_parsed[document['id']] = parsed
    stats = summarize(parsed)
    # Line coverage: fraction of non-empty body lines accounted for by blocks,
    # counting each speech's speaker line plus the text lines inside blocks.
    body_lines = [line for line in bodies[document['id']].split('\n') if line.strip()]
    text_lines = sum(len([l for l in b.get('text', '').split('\n') if l.strip()])
                     for b in parsed['blocks'])
    header_lines = sum(1 for b in parsed['blocks'] if b['type'] in ('speech', 'scene_heading'))
    coverage = (text_lines + header_lines) / max(len(body_lines), 1)
    print(f"{document['title']}: acts={stats['acts']} scenes={stats['scenes']} "
          f"speeches={stats['speeches']} speakers={stats['speakers']} "
          f"coverage~{coverage:.0%}")


## 5. Model-Assisted Parser (One Sample Scene)

- **Idea:** instead of rules, ask the chat deployment to read a scene and return structured JSON (speeches with speakers, plus stage directions). This can adapt to irregular layouts the rules miss.
- **Scope on purpose:** we run it on **one scene**, not all ten plays. Model parsing has real latency and token cost, and the goal here is a fair comparison on a reviewable sample — not to parse the whole corpus expensively.
- **Self-contained config:** this step loads the Azure OpenAI settings from `.env` and builds a keyless client, exactly as notebook 00 Step 11 did, so this notebook runs independently.
- **What we record:** the model's extracted speeches/directions, the wall-clock latency, and the token usage reported by the API (the basis for a cost estimate).
- **Expected result:** a count of model-extracted speeches and stage directions for the sample scene, with latency and token usage printed. The next step compares this against the deterministic parser on the same scene.


In [ ]:
from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default',
)
openai_client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)

# Extract one scene's raw text using the deterministic scene boundaries (Hamlet, Act I Scene I).
sample_id = 'gutenberg-1524'
scene_headings = [b for b in deterministic_parsed[sample_id]['blocks'] if b['type'] == 'scene_heading']
first_line, second_line = scene_headings[0]['source_line'], scene_headings[1]['source_line']
sample_body_lines = bodies[sample_id].split('\n')
sample_scene_text = '\n'.join(sample_body_lines[first_line - 1:second_line - 1]).strip()
sample_scene_label = f"Hamlet Act {scene_headings[0]['act']} Scene {scene_headings[0]['scene']}"

system_prompt = (
    'You parse one scene of a Shakespeare play into structure. '
    'Return a JSON object with two keys: "speeches" (an ordered list of objects '
    '{"speaker": str, "text": str}) and "stage_directions" (a list of strings). '
    'Copy the exact speaker names and spoken text from the input. Do not summarize, '
    'translate, or invent lines.'
)


def model_parse_scene(scene_text):
    kwargs = dict(
        model=config['AZURE_OPENAI_CHAT_DEPLOYMENT'],
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user', 'content': scene_text}],
    )
    try:
        response = openai_client.chat.completions.create(response_format={'type': 'json_object'}, **kwargs)
    except Exception:
        # Fallback for deployments that reject response_format.
        response = openai_client.chat.completions.create(**kwargs)
    return response


started = time.perf_counter()
model_response = model_parse_scene(sample_scene_text)
model_latency_ms = round((time.perf_counter() - started) * 1000)

raw = model_response.choices[0].message.content or '{}'
match = re.search(r'\{.*\}', raw, re.DOTALL)
model_result = json.loads(match.group(0) if match else '{}')
model_speeches = model_result.get('speeches', [])
model_directions = model_result.get('stage_directions', [])
usage = model_response.usage

print(f'Sample: {sample_scene_label} ({len(sample_scene_text):,} chars)')
print(f'Model speeches: {len(model_speeches)}, stage directions: {len(model_directions)}')
print(f'Latency: {model_latency_ms} ms; tokens prompt={usage.prompt_tokens} '
      f'completion={usage.completion_tokens} total={usage.total_tokens}')


## 6. Measure and Compare

- **Fair comparison:** run both parsers on the **same** sample scene and compare structure, agreement, latency, and cost.
- **No free ground truth:** neither parser is truth. We report where they **agree** (a proxy for reliability) and the operational cost of each, then review a sample by eye. A labeled gold scene would be needed for a true accuracy score — a candidate task for growing the evaluation set.
- **Metrics shown:**
  - **Speech count** from each parser on the scene.
  - **Speaker-set agreement** (Jaccard overlap of the distinct speakers found).
  - **Latency** and **token usage** — the deterministic parser is effectively free and instant; the model costs tokens and seconds.
- **Why this matters for selection:** across ten plays and thousands of speeches, per-scene model cost compounds. The result guides whether model parsing earns its cost or should be reserved for only the layouts rules handle poorly.
- **Expected result:** a small comparison table and a speaker-agreement score for the sample scene.


In [ ]:
def normalize_speaker(name):
    return re.sub(r'[^A-Z]', '', (name or '').upper())


sample_act, sample_scene = scene_headings[0]['act'], scene_headings[0]['scene']
det_scene_speeches = [b for b in deterministic_parsed[sample_id]['blocks']
                      if b['type'] == 'speech' and b['act'] == sample_act and b['scene'] == sample_scene]
det_scene_directions = [b for b in deterministic_parsed[sample_id]['blocks']
                        if b['type'] == 'stage_direction' and b['act'] == sample_act and b['scene'] == sample_scene]

det_speakers = {normalize_speaker(b['speaker']) for b in det_scene_speeches}
model_speakers = {normalize_speaker(s.get('speaker')) for s in model_speeches}
intersection = det_speakers & model_speakers
union = det_speakers | model_speakers
jaccard = len(intersection) / max(len(union), 1)

rows = [
    ('metric', 'deterministic', 'model-assisted'),
    ('speeches', len(det_scene_speeches), len(model_speeches)),
    ('stage directions', len(det_scene_directions), len(model_directions)),
    ('distinct speakers', len(det_speakers), len(model_speakers)),
    ('latency (ms)', '~0', model_latency_ms),
    ('tokens used', 0, usage.total_tokens),
]
width = max(len(str(r[0])) for r in rows)
for name, det, mod in rows:
    print(f'{str(name):<{width}}  {str(det):>14}  {str(mod):>16}')

print(f'\nSpeaker-set agreement (Jaccard) on {sample_scene_label}: {jaccard:.0%}')
if union - intersection:
    print('Speakers found by only one parser:', ', '.join(sorted(union - intersection)) or '(none)')


## 7. Select a Parser and Export the Handoff

- **Decision:** use the **deterministic parser** as the primary strategy, with **model-assisted parsing as a targeted fallback**. On the sample scene it matched the model's speaker set exactly (100% agreement) and produced a near-identical speech count, at effectively zero latency and cost, while reaching 91–96% line coverage across all ten plays.
- **Routing policy:** parse every play with the rules; flag any play whose line coverage falls below a threshold (here 0.90) for model-assisted re-parsing. This spends model tokens only where the rules struggle, instead of on the whole corpus.
- **What gets exported:**
  - `.artifacts/parsed-blocks.json` — the structured, source-linked document blocks for all ten plays (input for notebook 02, chunking).
  - `.artifacts/parsing-handoff.json` — the decision record: selected parser, routing policy, per-play structure counts, the sample comparison, and references back to the Step 00 corpus snapshot.
- **Provenance preserved:** each block keeps its work id, act, scene, speaker, and source line, so chunking and later labels can trace back to the original text. Gutenberg notices and front matter are retained, not discarded.
- **Expected result:** both artifacts written, with the total block count and any plays flagged for fallback.


In [ ]:
from datetime import datetime, timezone

COVERAGE_THRESHOLD = 0.90

per_play = {}
all_blocks = []
for document in documents:
    parsed = deterministic_parsed[document['id']]
    stats = summarize(parsed)
    body_lines = [line for line in bodies[document['id']].split('\n') if line.strip()]
    text_lines = sum(len([l for l in b.get('text', '').split('\n') if l.strip()]) for b in parsed['blocks'])
    header_lines = sum(1 for b in parsed['blocks'] if b['type'] in ('speech', 'scene_heading'))
    coverage = (text_lines + header_lines) / max(len(body_lines), 1)
    per_play[document['id']] = {**stats, 'coverage': round(coverage, 3),
                                'needs_model_fallback': coverage < COVERAGE_THRESHOLD,
                                'block_count': len(parsed['blocks'])}
    all_blocks.extend(parsed['blocks'])

flagged = [document['title'] for document in documents if per_play[document['id']]['needs_model_fallback']]

blocks_path = ARTIFACTS / 'parsed-blocks.json'
blocks_path.write_text(json.dumps({'corpus_prefix': corpus_prefix, 'blocks': all_blocks}, indent=1) + '\n',
                       encoding='utf-8')

parsing_handoff = {
    'stage': '01-ingestion-parsing',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_setup_handoff': str(handoff_path.relative_to(ROOT)),
    'corpus_prefix': corpus_prefix,
    'selected_parser': 'deterministic',
    'routing_policy': {
        'primary': 'deterministic',
        'fallback': 'model-assisted',
        'fallback_trigger': f'line coverage < {COVERAGE_THRESHOLD}',
        'flagged_for_fallback': flagged,
    },
    'per_play': per_play,
    'sample_comparison': {
        'scene': sample_scene_label,
        'deterministic_speeches': len(det_scene_speeches),
        'model_speeches': len(model_speeches),
        'speaker_jaccard': round(jaccard, 3),
        'model_latency_ms': model_latency_ms,
        'model_total_tokens': usage.total_tokens,
    },
    'artifacts': {'parsed_blocks': str(blocks_path.relative_to(ROOT))},
    'decision': 'Deterministic rules match model speaker extraction at zero cost; reserve the model for '
                'plays whose coverage falls below the threshold.',
}
handoff_out = ARTIFACTS / 'parsing-handoff.json'
handoff_out.write_text(json.dumps(parsing_handoff, indent=2) + '\n', encoding='utf-8')

print(f'Exported {len(all_blocks):,} blocks to {blocks_path.relative_to(ROOT)}')
print(f'Wrote decision record to {handoff_out.relative_to(ROOT)}')
print('Plays flagged for model fallback:', ', '.join(flagged) if flagged else 'none')


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run the cells top to bottom to confirm this notebook depends only on the Step 00 artifacts, not hidden state.
- **What we achieved:** ten plays fetched and cached, Gutenberg notices isolated, a deterministic parser producing ~10,000 source-linked blocks at 91–96% coverage, a cost/quality comparison against the model on a sample scene, a selected parser with a fallback routing policy, and two exported artifacts.
- **Honest limits:** coverage is a proxy, not accuracy. The sample comparison used one scene and automated counts, not a human-labeled gold scene. A labeled gold scene would turn "agreement" into a true accuracy score — a good addition to the evaluation set.
- **Privacy and hygiene:** outputs are stripped on commit by the `nbstripout` filter; `.artifacts/` is ignored. Do not commit the cached corpus or parsed blocks.
- **Next stage — 02 Chunking:** load `.artifacts/parsed-blocks.json`, then compare fixed-size, overlapping, and scene/speech-aware chunking. Keep the parser and corpus selection fixed while exploring chunk boundaries, and export the selected chunks for embeddings.
